<a href="https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/assignments/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Because the problem that I chose is a "whoch first?" ranking - identifying the pages that need a content refresh, I would choose classifiers that output probabilites (which was also shown in the SKILLS file) to sort the queue, evaluated by Precision@K.


I would start with a simple classifier and then go to the more complex ones:

- LR: linear baseline, more understandable decision to ensure our features have basic predictive power
- RF: to add non-linerity, more complex model with robustness against outliers
- Gradient Boosting: for example XGBoost, CatBoost, LightGBM: to capture complex interactions between ipressions, positions and CTRs to maximize our ranking metric

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

To ensure good generalization to new, unsen websites I would go with a Grouped Split (GroupShuffleSplit) grouped by client_hash_id. This prevents data leakage. What is also important is the fact thah client_has_id cannot be in the features that we use for modelling.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

TARGET_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

In [ ]:
query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_past,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_past,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN NULLIF(gsc_avg_position, 0) END) AS avg_pos_past,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_future
    FROM read_parquet('{TARGET_MONTH}')
    GROUP BY 1, 2
    HAVING imp_past >= 100
"""
df = con.sql(query).df()

# Calculate CTR (Clicks / Impressions * 100, matching the x100 percentage format)
df['ctr_past'] = np.where(df['imp_past'] > 0, (df['clicks_past'] / df['imp_past']) * 100, 0)
df['is_declining_label'] = (df['imp_future'] < 0.8 * df['imp_past']).astype(int)

df = df.dropna(subset=['avg_pos_past']).copy()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
features = ['imp_past', 'ctr_past', 'avg_pos_past']
target = 'is_declining_label'

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id']))

X_train = df.iloc[train_idx][features]
y_train = df.iloc[train_idx][target]
X_val = df.iloc[val_idx][features]
y_val = df.iloc[val_idx][target]

#validation set
df_val = df.iloc[val_idx].copy()

print(f"Training: {len(X_train)} rows, Validation: {len(X_val)} rows")
print(f"Unique clients in training set: {df.iloc[train_idx]['client_hash_id'].nunique()}")
print(f"Unique clients in validation set: {df_val['client_hash_id'].nunique()}")

Training: 61747 rows, Validation: 15793 rows
Unique clients in training set: 28
Unique clients in validation set: 10


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

To ensure a honest comparison, I first evaluate the Week-4 baseline rule on the new validation set (df_val). Then, I train three models (Logistic Regression, Random Forest, and XGBoost) on the training set and evaluate their ranking performance (Precision@50) on the same validation set. The final output is the comparison table.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
#base line rule
df_val['is_edge_of_page_1'] = ((df_val['avg_pos_past'] >= 7) & (df_val['avg_pos_past'] <= 10)).astype(int)
df_val['is_low_ctr'] = (df_val['ctr_past'] < 1.0).astype(int)
df_val['has_decent_volume'] = (df_val['imp_past'] >= 500).astype(int)
df_val['is_at_risk'] = df_val['has_decent_volume'] * df_val['is_edge_of_page_1'] * df_val['is_low_ctr']
df_val['baseline_score'] = np.where(df_val['is_at_risk'] > 0, 1.0 / (df_val['ctr_past'] + 0.01), 0)

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_val.mean()
baseline_p50 = precision_at_k(y_val, df_val['baseline_score'], k=50)

In [ ]:
#models
lr = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_val)[:, 1]
lr_p50 = precision_at_k(y_val, lr_scores, k=50)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_val)[:, 1]
rf_p50 = precision_at_k(y_val, rf_scores, k=50)

xgb = XGBClassifier(n_estimators = 100, random_state=42, max_depth=4, learning_rate = 0.1)
xgb.fit(X_train, y_train)
xgb_scores = xgb.predict_proba(X_val)[:, 1]
xgb_p50 = precision_at_k(y_val, xgb_scores, k=50)

In [ ]:
#comparison
comparison_table = pd.DataFrame({
    'Model': ['Baseline', 'Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision@50': [baseline_p50, lr_p50, rf_p50, xgb_p50]
})
comparison_table.to_markdown(index = False)

'| Model               |   Precision@50 |\n|:--------------------|---------------:|\n| Baseline            |           0.2  |\n| Logistic Regression |           0.1  |\n| Random Forest       |           0.18 |\n| XGBoost             |           0.18 |'

In [ ]:
comparison_table

,Model,Precision@50
0,Baseline,0.20
1,Logistic Regression,0.10
2,Random Forest,0.18
3,XGBoost,0.18


In [ ]:
lr_scores

array([2.65926848e-01, 3.59026680e-01, 3.08268395e-01, ...,
       3.51909076e-01, 2.58494366e-01, 2.96197002e-09])

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The comparison table shows that the chosen ML models failed to beat the Baseline rule at Precision@50.

Why did that happen? In my opinion it is because of this factors:

- Global vs. Top-K Optimization: The ML classifiers optimize for global metrics across the entire dataset (~61,000 rows). In contrast, the baseline rule was hardcoded to target a very specific business vulnerability (pages on the edge of Page 1 with low CTR) and push them to the absolute top of the queue. The models prioritized general patterns over these specific top-ranking edge cases.

- Lack of Non-Linear Features: Logistic Regression performed the worst (0.10) because it assumes a linear relationship. It doesn't understand that positions 7-10 form a specific "danger zone."

Next Steps:
This honest evaluation proves that simply throwing complex algorithms at raw features is not enough. To make the ML models win, I have to must expand the feature space. For example: the engineered flags (like is_edge_of_page_1) and additional context from the dataset (e.g., keyword data presence, time-based trends, and staleness) so they can find multidimensional signals that a human rule cannot.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.